In [1]:
%%capture
import subprocess
def run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'OK' if r.returncode==0 else 'ERR'} | {label or cmd[:60]}")
    if r.returncode != 0: print(f"   {r.stderr[-200:]}")

run("pip install fastapi uvicorn pyngrok nest-asyncio", "fastapi + ngrok")
run("pip install chromadb sentence-transformers", "chroma + embeddings")
run("pip install PyMuPDF python-docx", "pymupdf + docx")
run("pip install transformers accelerate bitsandbytes", "transformers")
run("pip install gdown", "gdown")


In [2]:
%%capture
import os, gdown, glob
from pathlib import Path

# ── Dossier de destination ────────────────────────────────────────────────────
CONTRACTS_DIR = Path('/kaggle/working/contracts')
CONTRACTS_DIR.mkdir(exist_ok=True)

# ── Votre Google Drive folder ID ─────────────────────────────────────────────
# Extrait depuis: https://drive.google.com/drive/folders/12pMw4B4H63GxqEjVtTsAWzAiFaeQcgCP
FOLDER_ID = '12pMw4B4H63GxqEjVtTsAWzAiFaeQcgCP'


try:
    gdown.download_folder(
        id=FOLDER_ID,
        output=str(CONTRACTS_DIR),
        quiet=False,
        use_cookies=False
    )
except Exception as e:
    print(f'gdown error: {e}')
    print('Tentative avec URL directe...')
    url = f'https://drive.google.com/drive/folders/{FOLDER_ID}'
    gdown.download_folder(url=url, output=str(CONTRACTS_DIR), quiet=False)

# ignore locked files
all_files = [f for f in list(CONTRACTS_DIR.rglob('*.pdf')) + \
            list(CONTRACTS_DIR.rglob('*.docx')) + \
            list(CONTRACTS_DIR.rglob('*.txt'))
            if not f.name.startswith(('~$', 'عقد بيع شقة سكنية', 'عقد عمل', 'رسالة مفتوحة إلى السيد والي الجهة الشرقية',
                                      'عقد إيجار منقول' , 'شكاية إلى السيد الوالي من اجل رفع الضرر' ,'عقد تأجير شقة مفروشة',
                                     'رهههن',
                                      # 'نموذج - عقد بيع شقة سكنية',
                                     'نمودج إشهاد بالتنازل عن شكاية',
                                      'عقد رهن محل تجاري',
                                     ))]


In [3]:
import re, hashlib, threading, asyncio
import fitz
import chromadb
import torch
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

DB_DIR = '/kaggle/working/chromadb'

print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

✅ Imports OK
GPU: True
VRAM: 15.6 GB


In [5]:
def normalize_arabic(text):
    text = re.sub(r'ـ+', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_pdf_text(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return '\n'.join(lines)

ARTICLE_RE = re.compile(
    r'(البند\s+(?:الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر'
    r'|[\d\u0660-\u0669]+)'
    r'|المادة\s+(?:الأولى|الثانية|[\d\u0660-\u0669]+)'
    r'|الفصل\s+(?:الأول|الثاني|[\d\u0660-\u0669]+)'
    r'|أولاً|ثانياً|ثالثاً|رابعاً|خامساً)',
    re.UNICODE
)

CONTRACT_KEYWORDS = {
    'عقد_إيجار':  ['إيجار', 'مستأجر', 'مؤجر', 'كراء', 'مكتري', 'مكري'],
    'عقد_بيع':    ['بيع', 'مشتري', 'بائع', 'ثمن', 'ملكية'],
    'عقد_عمل':    ['عمل', 'عامل', 'راتب', 'أجر', 'أجير', 'مشغل'],
    'عقد_شراكة':  ['شراكة', 'شريك', 'حصة', 'أرباح', 'شركة'],
    'عقد_مقاولة': ['مقاولة', 'مقاول', 'أشغال', 'بناء'],
    'عقد_قرض':    ['قرض', 'مقترض', 'مقرض', 'فائدة', 'سلفة'],
}

def detect_contract_type(text):
    sample = normalize_arabic(text[:3000])
    scores = {t: sum(kw in sample for kw in kws) for t, kws in CONTRACT_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else 'عقد_عام'

def parse_pdf(path):
    doc = fitz.open(str(path))
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text('text', flags=fitz.TEXT_PRESERVE_WHITESPACE)
        if text.strip():
            pages.append(f'[صفحة {i+1}]\n{text}')
    doc.close()
    return clean_pdf_text('\n\n'.join(pages))

def chunk_contract(text, max_size=1200, min_size=150):
    parts = ARTICLE_RE.split(text)
    chunks = []
    if len(parts) > 3:
        cur_art, cur_txt = 'مقدمة', ''
        for part in parts:
            if not part: continue
            if ARTICLE_RE.match(part):
                if len(cur_txt.strip()) >= min_size:
                    chunks.append({'text': cur_txt.strip(), 'article': cur_art})
                cur_art, cur_txt = part.strip(), part
            else:
                cur_txt += ' ' + part
                if len(cur_txt) > max_size:
                    chunks.append({'text': cur_txt[:max_size].strip(), 'article': cur_art})
                    cur_txt = cur_txt[max_size:]
        if len(cur_txt.strip()) >= min_size:
            chunks.append({'text': cur_txt.strip(), 'article': cur_art})
    else:
        step = max_size // 6
        words = text.split()
        for i in range(0, len(words), step - step//5):
            chunk = ' '.join(words[i:i+step])
            if len(chunk) >= min_size:
                chunks.append({'text': chunk, 'article': f'قسم_{i//step+1}'})
    return chunks

print("✅ Arabic utils OK")

✅ Arabic utils OK


In [6]:
print("Chargement du modèle d'embedding...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print("✅ Embedding model chargé")

client_db = chromadb.PersistentClient(path=DB_DIR)
try:
    collection = client_db.get_collection('moroccan_contracts')
    print(f"✅ Collection existante chargée: {collection.count()} chunks")
except Exception:
    collection = client_db.create_collection(
        name='moroccan_contracts',
        metadata={'hnsw:space': 'cosine'}
    )
    print("✅ Nouvelle collection créée")

Chargement du modèle d'embedding...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Embedding model chargé
✅ Nouvelle collection créée


In [7]:
# Copiez vos PDFs dans /kaggle/working/contracts/ d'abord
# (via gdown, upload manuel, ou dataset Kaggle)

all_files = list(CONTRACTS_DIR.rglob('*.pdf')) + \
            list(CONTRACTS_DIR.rglob('*.docx'))

print(f"Fichiers trouvés: {len(all_files)}")

if collection.count() == 0 and all_files:
    print("Indexation en cours...")
    for fpath in all_files:
        print(f"  → {fpath.name}")
        if fpath.suffix == '.pdf':
            text = parse_pdf(fpath)
        else:
            from docx import Document as DocxDoc
            text = '\n'.join(p.text for p in DocxDoc(str(fpath)).paragraphs if p.text.strip())
        
        if len(text) < 100:
            continue
        
        ctype = detect_contract_type(text)
        chunks = chunk_contract(text)
        fhash = hashlib.md5(fpath.read_bytes()).hexdigest()[:8]
        
        texts_to_embed = [f'passage: {normalize_arabic(c["text"])}' for c in chunks]
        embeddings = embed_model.encode(texts_to_embed, batch_size=8, normalize_embeddings=True)
        
        ids, docs, metas, embeds = [], [], [], []
        for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
            ids.append(f'{fhash}_{i}')
            docs.append(chunk['text'])
            metas.append({'file': fpath.name, 'contract_type': ctype, 'article': chunk['article']})
            embeds.append(emb.tolist())
        
        for i in range(0, len(ids), 50):
            collection.add(ids=ids[i:i+50], documents=docs[i:i+50],
                           metadatas=metas[i:i+50], embeddings=embeds[i:i+50])
        print(f"     {ctype} | {len(chunks)} chunks")

print(f"\n✅ Total indexé: {collection.count()} chunks")


Fichiers trouvés: 48
Indexation en cours...
  → 3.pdf
     عقد_عام | 3 chunks
  → عقد إيجار شقة سكنية.pdf
     عقد_عام | 4 chunks
  → نموذج عقد بيع محل تجاري صيغة و نموذج عقد بيع محل تجاري.docx
     عقد_بيع | 3 chunks
  → عقد بيع سيارة.docx
     عقد_بيع | 3 chunks
  → اعتراف بالدين والتزام بالأداء.docx
     عقد_عام | 1 chunks
  → شكاية إلى السيد الوالي من اجل رفع الضرر.docx
     عقد_عام | 1 chunks
  → الزام عن افراغ محل بعد 6اشهر.docx
     عقد_عام | 1 chunks
  → نموذج - عقد بيع شقة سكنية.docx
     عقد_بيع | 4 chunks
  → التصريح بالشرف العزوبة.docx
     عقد_عام | 1 chunks
  → ~$د كراء السعيدي عبد الغني.docx


PackageNotFoundError: Package not found at '/kaggle/working/contracts/~$د كراء السعيدي عبد الغني.docx'

In [ ]:
app = FastAPI(title="Moroccan RAG Contract Server", version="1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class RetrieveRequest(BaseModel):
    query: str
    contract_type: str = None
    n: int = 5

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 2500

class HealthResponse(BaseModel):
    status: str
    chunks_indexed: int
    gpu_available: bool
    model_loaded: bool

@app.get("/health", response_model=HealthResponse)
def health():
    return {
        "status": "ok",
        "chunks_indexed": collection.count(),
        "gpu_available": torch.cuda.is_available(),
        "model_loaded": llm_model is not None,
    }

@app.post("/retrieve")
def retrieve_endpoint(req: RetrieveRequest):
    if collection.count() == 0:
        return {"chunks": [], "total": 0}
    
    q_emb = embed_model.encode(
        f'query: {normalize_arabic(req.query)}',
        normalize_embeddings=True
    ).tolist()
    
    where = None
    if req.contract_type and req.contract_type not in ('عقد_عام', ''):
        where = {'contract_type': req.contract_type}
    
    n_real = min(req.n, collection.count())
    results = collection.query(
        query_embeddings=[q_emb], n_results=n_real,
        where=where, include=['documents', 'metadatas', 'distances']
    )
    chunks = [
        {'text': doc, 'meta': meta, 'score': round(1 - dist, 3)}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )
    ]
    return {"chunks": chunks, "total": collection.count()}

@app.post("/generate")
def generate_endpoint(req: GenerateRequest):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': req.prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(llm_model.device)
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=req.max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return {"contract": tokenizer.decode(generated, skip_special_tokens=True)}

print("✅ FastAPI app définie")

In [1]:
# ⚠️  Mettez votre token ngrok ici :
# https://dashboard.ngrok.com/get-started/your-authtoken
from kaggle_secrets import UserSecretsClient()

user_secrets = UserSecretsClient()
NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
PORT = 8000 

ngrok.set_auth_token(NGROK_TOKEN)

# Ouvre le tunnel
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url
print("=" * 60)
print(f"🌐 API publique : {public_url}")
print(f"📋 Health check : {public_url}/health")
print(f"📖 Docs Swagger : {public_url}/docs")
print("=" * 60)
print("⚠️  Copiez cette URL dans Streamlit (sidebar > URL Kaggle API)")
print("    Le serveur tourne tant que ce notebook est actif.")

# Lance uvicorn dans un thread séparé
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print("✅ Serveur démarré !")

NameError: name 'ngrok' is not defined